In [1]:
import os
import json
import numpy as np
from tqdm import tqdm

BASE_RAW_DIR = "../test2/all_raw_files"
BASE_BBOX_DIR = "../test_retry/n/testdetected_bboxes"
ANNOT_DIR = "../test2/test_annotations"
LOG_PATH = "missing_inf_dp.log"

DISP_WIDTH = 256
DISP_HEIGHT = 105
RIGHT_HEIGHT = 420
RIGHT_WIDTH = 1000

def load_disparity_map_np(path):
    with open(path, "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.uint8)
    data = data.reshape(-1, 2)
    disp_flat = data[:, 0] + data[:, 1] / 256.0
    return disp_flat.reshape(DISP_HEIGHT, DISP_WIDTH)

def bbox_to_mode_distance_np(bbox, disp_map, inf_dp):
    x1, y1, x2, y2 = map(int, bbox)
    i1, i2 = x1 // 4, x2 // 4
    j1, j2 = (RIGHT_HEIGHT - y2) // 4, (RIGHT_HEIGHT - y1) // 4
    i1, i2 = max(i1, 0), min(i2, DISP_WIDTH)
    j1, j2 = max(j1, 0), min(j2, DISP_HEIGHT)
    
    region = disp_map[j1:j2, i1:i2].flatten()
    region = region[region > inf_dp]  # 有効視差のみ
    if len(region) == 0:
        return None

    distances = 560 / (region - inf_dp)
    distances = distances[np.isfinite(distances) & (distances < 200)]

    if len(distances) == 0:
        return None

    bins = np.round(distances * 10).astype(int)  # 0.1m単位でbinning
    mode_bin = np.bincount(bins).argmax()
    return mode_bin / 10.0

def find_center_bbox(bboxes):
    cx = RIGHT_WIDTH / 2
    return min(bboxes, key=lambda b: abs((b["bbox"][0] + b["bbox"][2]) / 2 - cx))

def process_scene(scene_id, log_lines):
    result = {}
    bbox_dir = os.path.join(BASE_BBOX_DIR, scene_id)
    raw_dir = os.path.join(BASE_RAW_DIR, scene_id)
    annot_path = os.path.join(ANNOT_DIR, f"{scene_id}.json")

    if not (os.path.exists(bbox_dir) and os.path.exists(raw_dir) and os.path.exists(annot_path)):
        return {}

    try:
        with open(annot_path) as f:
            annot = json.load(f)
            inf_dp = annot["sequence"][0]["inf_DP"]
    except Exception:
        log_lines.append(scene_id)
        return {}

    for frame_file in sorted(os.listdir(bbox_dir)):
        if not frame_file.endswith(".json"):
            continue
        frame_base = frame_file.replace(".json", "")
        frame_num = frame_base.replace("frame_", "")
        raw_base = frame_num.zfill(8) + "f"
        raw_path = os.path.join(raw_dir, f"{raw_base}.raw")
        bbox_path = os.path.join(bbox_dir, frame_file)

        if not (os.path.exists(raw_path) and os.path.exists(bbox_path)):
            continue

        with open(bbox_path) as f:
            bboxes = json.load(f)
        if not isinstance(bboxes, list):
            bboxes = [bboxes]
        if not bboxes:
            continue

        target = find_center_bbox(bboxes)
        disp_map = load_disparity_map_np(raw_path)
        mode_dist = bbox_to_mode_distance_np(target["bbox"], disp_map, inf_dp)

        if mode_dist is not None:
            result[frame_base] = mode_dist
    return result

# 全体処理
all_results = {}
missing_scenes = []
scene_ids = sorted(os.listdir(BASE_BBOX_DIR))

for sid in tqdm(scene_ids):
    r = process_scene(sid, missing_scenes)
    if r:
        all_results[sid] = r

with open("testdistance_estimates.json", "w") as f:
    json.dump(all_results, f, indent=2)
with open(LOG_PATH, "w") as f:
    f.write("\n".join(missing_scenes))

print(f"✅ 完了: {len(all_results)} シーン処理 → distance_estimates.json に保存")
print(f"📝 inf_DP 取得エラーのシーン: {len(missing_scenes)} 件 → missing_inf_dp.log に記録")


100%|██████████| 239/239 [00:08<00:00, 28.50it/s]


✅ 完了: 239 シーン処理 → distance_estimates.json に保存
📝 inf_DP 取得エラーのシーン: 0 件 → missing_inf_dp.log に記録


In [2]:
import os
import json
import matplotlib.pyplot as plt

# 読み込み
with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

# 出力ディレクトリ
output_dir = "./scene"
os.makedirs(output_dir, exist_ok=True)

for scene_id, frame_distances in data.items():
    # フレーム番号順に並べ替え
    frames = sorted(frame_distances.keys())
    distances = [frame_distances[frame] for frame in frames]

    # プロット
    plt.figure()
    plt.plot(range(1, len(distances)+1), distances, marker='o')
    plt.title(f"Scene {scene_id} - Distance Estimate")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)

    # 保存
    save_path = os.path.join(output_dir, f"{scene_id}.png")
    plt.savefig(save_path)
    plt.close()

print("✅ 各シーンのグラフを test/outputs/distance_graphs/ に保存しました。")


✅ 各シーンのグラフを test/outputs/distance_graphs/ に保存しました。
